# 结构化输出
结构化输出可让智能体以特定、可预测的格式返回数据。无需解析自然语言回复，就能直接获取 JSON 对象、Pydantic 模型或数据类格式的结构化数据，供应用程序直接调用。

LangChain 的 `create_agent` 函数可自动处理结构化输出。用户只需设定所需的结构化输出模式，当模型生成结构化数据后，该数据会被捕获、校验，并最终存入智能体状态的 `structured_response` 键中。

```
def create_agent(
    ...
    response_format: Union[
        ToolStrategy[StructuredResponseT],
        ProviderStrategy[StructuredResponseT],
        type[StructuredResponseT],
        None,
    ]
```

## 1. 响应格式(Response format)
用于控制智能体返回结构化数据的方式：`response_format`
- `ToolStrategy[StructuredResponseT]`：借助工具调用实现结构化输出
- `ProviderStrategy[StructuredResponseT]`：采用服务商原生的结构化输出方案
- `type[StructuredResponseT]`：模式类型 —— 基于模型能力自动选择最优策略
- `None`：未显式请求结构化输出

当直接传入模式类型时，LangChain 会自动进行选择：
- 若所选的模型及服务商支持原生结构化输出（例如 OpenAI、Anthropic（Claude）或 xAI（Grok）），则采用 `ProviderStrategy`；
- 对于其他所有模型，则采用 `ToolStrategy`。

结构化响应结果会存入智能体最终状态的 `structured_response` 键中。

## 2. 服务商原生策略(Provider strategy)
部分模型服务商通过其 API 原生支持结构化输出（例如 OpenAI、xAI（Grok）、Gemini、Anthropic（Claude））。在支持该功能的情况下，这是可靠性最高的实现方式。

若要使用此策略，需要配置一个 `ProviderStrategy`。
```
class ProviderStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    strict: bool | None = None
```

`schema`必填项

该参数用于定义结构化输出的数据格式，支持以下类型：
- **Pydantic models**：具备字段校验能力的 BaseModel 子类，返回经过校验的 Pydantic 实例。
- **Dataclasses**：带有类型注解的 Python 数据类，返回字典格式数据。
- **TypedDict**：类型字典类，返回字典格式数据。
- **JSON Schema**：符合 JSON 模式规范的字典，返回字典格式数据。

`strict`可选布尔类型参数，

用于启用严格模式的模式校验。该功能受部分服务商支持（例如 OpenAI、xAI）。默认值为 None（即禁用状态）。

当将模式类型直接传入`create_agent`的`response_format`参数并指定为`ProviderStrategy`，且模型支持原生结构化输出时，LangChain 会自动启用该策略。

In [6]:
# pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

model = ChatOllama(model="qwen3:1.7b")

agent = create_agent(
    model=model,
    response_format=ContactInfo  # Auto-selects ProviderStrategy 其实没有选择到ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [5]:
# dataclass
from dataclasses import dataclass
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

model = ChatOllama(model="qwen3:1.7b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo  # Auto-selects ProviderStrategy 其实没有选择到ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [5]:
# TyoeDict
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

model = ChatOllama(model="qwen3:1.7b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ContactInfo  # Auto-selects ProviderStrategy 其实没有选择到ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [14]:
# JSON Schema
from langchain.agents import create_agent
from langchain_ollama import ChatOllama
from langchain.agents.structured_output import ProviderStrategy

contact_info_schema = {
    "type": "object",
    "description": "Contact information for a person.",
    "properties": {
        "name": {"type": "string", "description": "The name of the person"},
        "email": {"type": "string", "description": "The email address of the person"},
        "phone": {"type": "string", "description": "The phone number of the person"}
    },
    "required": ["name", "email", "phone"]
}

model = ChatOllama(model="qwen3:1.7b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ProviderStrategy(contact_info_schema) # Ollama不支持原生结构化输出
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

服务商原生结构化输出具备高可靠性与严格的校验机制，因为模型服务商会对数据模式进行强制约束。建议在支持该功能的场景下使用此方案。
> 若所选模型的服务商原生支持结构化输出，则编写`response_format=ProductReview`与编写`response_format=ProviderStrategy(ProductReview)`在功能上完全等效。
> 无论采用上述哪种写法，若模型不支持结构化输出，智能体都会回退至工具调用策略执行。
## 3. 工具调用策略(Tool calling strategy)
对于不支持原生结构化输出的模型，LangChain 会通过工具调用实现相同的输出效果。该策略适用于所有支持工具调用功能的模型（主流的现代模型均支持）。

使用本策略时，需配置为：`ToolStrategy`
```
class ToolStrategy(Generic[SchemaT]):
    schema: type[SchemaT]
    tool_message_content: str | None
    handle_errors: Union[
        bool,
        str,
        type[Exception],
        tuple[type[Exception], ...],
        Callable[[Exception], str],
    ]
```
`schema` 必填项

用于定义结构化输出格式的数据结构规范，支持以下类型：
- **Pydantic models**：带字段校验的`BaseModel`子类，返回经过校验的 `Pydantic` 实例。
- **Dataclasses**：带类型注解的 Python 数据类，返回字典格式数据。
- **TypedDict**：类型字典类，返回字典格式数据。
- **JSON Schema**：符合 JSON 模式规范的字典，返回字典格式数据。
- 联合类型：多种模式可选，模型将根据上下文选择最合适的模式。

`tool_message_content`

生成结构化输出时，返回的工具消息所对应的自定义内容。若未配置该参数，将默认返回一条展示结构化响应数据的消息。

`handle_errors`

结构化输出校验失败的异常处理策略，默认值为True
- True：捕获所有异常，使用默认异常模板返回结果
- str：捕获所有异常，使用该自定义消息返回结果
- type[Exception]：仅捕获该指定类型的异常，使用默认消息返回结果
- tuple[type[Exception], ...]：仅捕获这些指定类型的异常，使用默认消息返回结果
- Callable[[Exception], str]：自定义函数，执行后返回异常提示消息
- False：不进行重试，让异常向上抛出

In [18]:
# Pydantic
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReview(rating=5, sentiment='negative', key_points=['5 out of 5 stars', 'Fast shipping', 'but expensive'])

In [6]:
# dataclasses 不理解为什么这么慢，回去跑一下 TODO
from dataclasses import dataclass
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

@dataclass
class ProductReview:
    """Analysis of a product review."""
    rating: int | None  # The rating of the product (1-5)
    sentiment: Literal["positive", "negative"]  # The sentiment of the review
    key_points: list[str]  # The key points of the review

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}

KeyboardInterrupt: 

In [2]:
# TypedDict
from typing import Literal
from typing_extensions import TypedDict
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

class ProductReview(TypedDict):
    """Analysis of a product review."""
    rating: int | None  # The rating of the product (1-5)
    sentiment: Literal["positive", "negative"]  # The sentiment of the review
    key_points: list[str]  # The key points of the review

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductReview)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}

KeyboardInterrupt: 

In [7]:
# JSON Schema
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

product_review_schema = {
    "type": "object",
    "description": "Analysis of a product review.",
    "properties": {
        "rating": {
            "type": ["integer", "null"],
            "description": "The rating of the product (1-5)",
            "minimum": 1,
            "maximum": 5
        },
        "sentiment": {
            "type": "string",
            "enum": ["positive", "negative"],
            "description": "The sentiment of the review"
        },
        "key_points": {
            "type": "array",
            "items": {"type": "string"},
            "description": "The key points of the review"
        }
    },
    "required": ["sentiment", "key_points"]
}

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(product_review_schema)
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# {'rating': 5, 'sentiment': 'positive', 'key_points': ['fast shipping', 'expensive']}

KeyboardInterrupt: 

In [17]:
# Union Type Pydantic因为有Filed设置description所以模型可以更好理解执行？ TODO
from pydantic import BaseModel, Field
from typing import Literal, Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

class ProductReview(BaseModel):
    """Analysis of a product review."""
    rating: int | None = Field(description="The rating of the product", ge=1, le=5)
    sentiment: Literal["positive", "negative"] = Field(description="The sentiment of the review")
    key_points: list[str] = Field(description="The key points of the review. Lowercase, 1-3 words each.")

class CustomerComplaint(BaseModel):
    """A customer complaint about a product or service."""
    issue_type: Literal["product", "service", "shipping", "billing"] = Field(description="The type of issue")
    severity: Literal["low", "medium", "high"] = Field(description="The severity of the complaint")
    description: str = Field(description="Brief description of the complaint")

model = ChatOllama(model="qwen3:4b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ProductReview, CustomerComplaint])
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Analyze this review: 'Great product: 5 out of 5 stars. Fast shipping, but expensive'"}]
})
result["structured_response"]
# ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

ProductReview(rating=5, sentiment='positive', key_points=['fast shipping', 'expensive'])

### 3.1 自定义工具消息内容
该参数支持自定义生成结构化输出时，出现在对话历史中的消息内容：`tool_message_content`

In [15]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

class MeetingAction(BaseModel):
    """Action items extracted from a meeting transcript."""
    task: str = Field(description="The specific task to be completed")
    assignee: str = Field(description="Person responsible for the task")
    priority: Literal["low", "medium", "high"] = Field(description="Priority level")

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
        schema=MeetingAction,
        tool_message_content="Action item captured and added to meeting notes!"
    )
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "From our meeting: Sarah needs to update the project timeline as soon as possible"}]
})

# 打印result中的每条消息的类型和内容
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

From our meeting: Sarah needs to update the project timeline as soon as possible
================================== Ai Message ==================================
Tool Calls:
  MeetingAction (47835da1-1649-44e4-ad9b-28ef405a2661)
 Call ID: 47835da1-1649-44e4-ad9b-28ef405a2661
  Args:
    task: update the project timeline
    assignee: Sarah
    priority: high
================================= Tool Message =================================
Name: MeetingAction

Action item captured and added to meeting notes!


### 3.2 异常处理
模型通过工具调用生成结构化输出时可能出现错误，LangChain 提供了智能重试机制，可自动处理此类错误。

#### 3.2.1多结构化输出错误
当模型错误调用多个结构化输出工具时，智能体会在工具消息中反馈错误信息，并提示模型重新尝试：

In [16]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

class ContactInfo(BaseModel):
    name: str = Field(description="Person's name")
    email: str = Field(description="Email address")

class EventDetails(BaseModel):
    event_name: str = Field(description="Name of the event")
    date: str = Field(description="Event date")

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(Union[ContactInfo, EventDetails])  # Default: handle_errors=True
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th
================================== Ai Message ==================================
Tool Calls:
  EventDetails (900ee026-6019-4a74-88c2-ecbbf4eefa57)
 Call ID: 900ee026-6019-4a74-88c2-ecbbf4eefa57
  Args:
    event_name: Tech Conference
    date: March 15th
================================= Tool Message =================================
Name: EventDetails

Returning structured response: event_name='Tech Conference' date='March 15th'


#### 3.2.2 模式校验错误
当结构化输出与预期模式不匹配时，智能体会反馈具体的错误信息：

In [19]:
# 本质还是提示词工程，看模型聪不聪明了
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain_ollama import ChatOllama

class ProductRating(BaseModel):
    rating: int | None = Field(description="Rating from 1-5", ge=1, le=5)
    comment: str = Field(description="Review comment")

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(ProductRating),  # Default: handle_errors=True
    system_prompt="You are a helpful assistant that parses product reviews. Do not make any field or value up."
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Parse this: Amazing product, 10/10!"}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Parse this: Amazing product, 10/10!
================================== Ai Message ==================================
Tool Calls:
  ProductRating (932f1fa3-a289-4c32-91f1-df5426270ff1)
 Call ID: 932f1fa3-a289-4c32-91f1-df5426270ff1
  Args:
    comment: Amazing product, 10/10!
    rating: 5
================================= Tool Message =================================
Name: ProductRating

Returning structured response: rating=5 comment='Amazing product, 10/10!'


#### 3.2.3 异常处理策略
可通过handle_errors参数自定义异常的处理方式：

##### 自定义异常提示信息：

In [20]:
ToolStrategy(
    schema=ProductRating,
    handle_errors="Please provide a valid rating between 1-5 and include a comment."
)

ToolStrategy(schema=<class '__main__.ProductRating'>, schema_specs=[_SchemaSpec(schema=<class '__main__.ProductRating'>, name='ProductRating', description='', schema_kind='pydantic', json_schema={'properties': {'rating': {'anyOf': [{'maximum': 5, 'minimum': 1, 'type': 'integer'}, {'type': 'null'}], 'description': 'Rating from 1-5', 'title': 'Rating'}, 'comment': {'description': 'Review comment', 'title': 'Comment', 'type': 'string'}}, 'required': ['rating', 'comment'], 'title': 'ProductRating', 'type': 'object'}, strict=None)], tool_message_content=None, handle_errors='Please provide a valid rating between 1-5 and include a comment.')

若`handle_errors`为字符串类型，智能体将始终通过固定的工具消息提示模型重新尝试。

##### 仅处理特定异常：

In [21]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=ValueError  # Only retry on ValueError, raise others
)

ToolStrategy(schema=<class '__main__.ProductRating'>, schema_specs=[_SchemaSpec(schema=<class '__main__.ProductRating'>, name='ProductRating', description='', schema_kind='pydantic', json_schema={'properties': {'rating': {'anyOf': [{'maximum': 5, 'minimum': 1, 'type': 'integer'}, {'type': 'null'}], 'description': 'Rating from 1-5', 'title': 'Rating'}, 'comment': {'description': 'Review comment', 'title': 'Comment', 'type': 'string'}}, 'required': ['rating', 'comment'], 'title': 'ProductRating', 'type': 'object'}, strict=None)], tool_message_content=None, handle_errors=<class 'ValueError'>)

若handle_errors为异常类型，仅当抛出的异常为指定类型时，智能体才会重新尝试（使用默认异常提示信息）；在其他所有情况下，该异常都会直接抛出。

##### 处理多种异常类型：


In [22]:
ToolStrategy(
    schema=ProductRating,
    handle_errors=(ValueError, TypeError)  # Retry on ValueError and TypeError
)

ToolStrategy(schema=<class '__main__.ProductRating'>, schema_specs=[_SchemaSpec(schema=<class '__main__.ProductRating'>, name='ProductRating', description='', schema_kind='pydantic', json_schema={'properties': {'rating': {'anyOf': [{'maximum': 5, 'minimum': 1, 'type': 'integer'}, {'type': 'null'}], 'description': 'Rating from 1-5', 'title': 'Rating'}, 'comment': {'description': 'Review comment', 'title': 'Comment', 'type': 'string'}}, 'required': ['rating', 'comment'], 'title': 'ProductRating', 'type': 'object'}, strict=None)], tool_message_content=None, handle_errors=(<class 'ValueError'>, <class 'TypeError'>))

若handle_errors为异常类型的元组，仅当抛出的异常为指定类型之一时，智能体才会重新尝试（使用默认异常提示信息）；其余所有情况下，该异常均会直接抛出。

##### 自定义异常处理函数：

In [23]:
from langchain.agents.structured_output import StructuredOutputValidationError
from langchain.agents.structured_output import MultipleStructuredOutputsError
from langchain_ollama import ChatOllama

def custom_error_handler(error: Exception) -> str:
    if isinstance(error, StructuredOutputValidationError):
        return "There was an issue with the format. Try again."
    elif isinstance(error, MultipleStructuredOutputsError):
        return "Multiple structured outputs were returned. Pick the most relevant one."
    else:
        return f"Error: {str(error)}"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[],
    response_format=ToolStrategy(
                        schema=Union[ContactInfo, EventDetails],
                        handle_errors=custom_error_handler
                    )  # Default: handle_errors=True
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract info: John Doe (john@email.com) is organizing Tech Conference on March 15th"}]
})

for msg in result['messages']:
    # If message is actually a ToolMessage object (not a dict), check its class name
    if type(msg).__name__ == "ToolMessage":
        print(msg.content)
    # If message is a dictionary or you want a fallback
    elif isinstance(msg, dict) and msg.get('tool_call_id'):
        print(msg['content'])

KeyboardInterrupt: 

##### No error handling:

In [24]:
response_format = ToolStrategy(
    schema=ProductRating,
    handle_errors=False  # All errors raised
)